# 02 · Claude score vs keyword score

Run this after `uv run jobscout-score` has scored some jobs. Scores are read from `data/scores.json`, so nothing here costs money.

In [ ]:
import json
import pandas as pd
from jobscout import store

jobs = store.load()
scores = pd.DataFrame(json.load(open("../data/scores.json")).values())

df = jobs.merge(scores, on="job_id")   # SQL: INNER JOIN ... ON job_id
df[["title", "employer", "score", "verdict"]].sort_values("score", ascending=False)

## Add the keyword baseline from notebook 01

In [ ]:
MY_KEYWORDS = ["azure", ".net", "fda", "iso 13485", "agile", "safe", "stakeholder"]

df["kw_score"] = df.description.str.lower().apply(lambda t: sum(k in t for k in MY_KEYWORDS))
df[["score", "kw_score"]].corr()

## Where do they disagree?

This is the interesting part. Read the postings at both extremes and decide who was right. If Claude is wrong, the fix is usually in `profile.md` or `SYSTEM_PROMPT`, not in code.

In [ ]:
df["gap"] = df.score - df.kw_score
cols = ["title", "employer", "score", "kw_score", "verdict"]
print("Claude likes, keywords don't:")
display(df.nlargest(3, "gap")[cols])
print("Keywords like, Claude doesn't:")
display(df.nsmallest(3, "gap")[cols])

In [ ]:
# Read one posting's concerns in full
row = df.sort_values("score").iloc[0]
row.title, row.concerns